In [2]:
import os

# Change to the parent directory (where data_raw is)
os.chdir('..')

print("Current directory:", os.getcwd())
print("\nFiles and folders here:")
for item in os.listdir():
    print(f"  {item}")

# Check if data_raw exists now
if os.path.exists('data_raw'):
    print("\n✅ data_raw folder exists!")
    print("Contents of data_raw:")
    for item in os.listdir('data_raw'):
        print(f"  {item}")
else:
    print("\n❌ data_raw folder NOT found")

Current directory: D:\IPL_Analytics_Project

Files and folders here:
  .git
  dashboard
  data_processed
  data_raw
  ipl_env
  notebooks
  README.md

✅ data_raw folder exists!
Contents of data_raw:
  deliveries.csv
  matches.csv


In [3]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("IPL CRICKET ANALYTICS DASHBOARD - DATA EXPLORATION")
print("=" * 70)

# Load datasets
deliveries = pd.read_csv('data_raw/deliveries.csv')
matches = pd.read_csv('data_raw/matches.csv')

print("\n✅ Data loaded successfully!")
print(f"\n📊 deliveries.csv: {deliveries.shape[0]:,} rows, {deliveries.shape[1]} columns")
print(f"📊 matches.csv: {matches.shape[0]:,} rows, {matches.shape[1]} columns")

# Quick preview
print("\n" + "=" * 70)
print("FIRST 3 ROWS - deliveries.csv")
print("=" * 70)
print(deliveries.head(3))

print("\n" + "=" * 70)
print("FIRST 3 ROWS - matches.csv")
print("=" * 70)
print(matches.head(3))

IPL CRICKET ANALYTICS DASHBOARD - DATA EXPLORATION

✅ Data loaded successfully!

📊 deliveries.csv: 260,920 rows, 17 columns
📊 matches.csv: 1,095 rows, 20 columns

FIRST 3 ROWS - deliveries.csv
   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball       batter   bowler  non_striker  batsman_runs  extra_runs  \
0     1   SC Ganguly  P Kumar  BB McCullum             0           1   
1     2  BB McCullum  P Kumar   SC Ganguly             0           0   
2     3  BB McCullum  P Kumar   SC Ganguly             0           1   

   total_runs extras_type  is_wicket player_dismissed dismissal_kind fielder  
0           1     legbyes          0              NaN            NaN     NaN  
1           0         NaN          

In [4]:
print("\n" + "=" * 70)
print("DATA CLEANING & FEATURE ENGINEERING")
print("=" * 70)

# Create clean copies
matches_clean = matches.copy()
deliveries_clean = deliveries.copy()

# Standardize season column
def extract_season_year(season):
    if '/' in str(season):
        return int(str(season).split('/')[0])
    else:
        return int(season)

matches_clean['season_year'] = matches_clean['season'].apply(extract_season_year)
matches_clean['season_name'] = matches_clean['season_year'].astype(str) + '-' + (matches_clean['season_year'] + 1).astype(str)

print("✅ Season standardization complete!")
print(f"Seasons: {sorted(matches_clean['season_year'].unique())}")

# Merge datasets
ipl_data = deliveries_clean.merge(
    matches_clean[['id', 'season_year', 'season_name', 'city', 'venue', 'winner', 'toss_winner', 'toss_decision']],
    left_on='match_id',
    right_on='id',
    how='left'
)

print(f"\n✅ Merged dataset shape: {ipl_data.shape[0]:,} rows, {ipl_data.shape[1]} columns")


DATA CLEANING & FEATURE ENGINEERING
✅ Season standardization complete!
Seasons: [np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

✅ Merged dataset shape: 260,920 rows, 25 columns


In [10]:
print("\n" + "=" * 70)
print("ADDING POWER PLAY & DEATH OVERS FLAGS (CORRECT CALCULATION)")
print("=" * 70)

# Sort the data properly by match, inning, ball sequence
# Create a sequential index within each match and inning
ipl_data = ipl_data.sort_values(['match_id', 'inning', 'ball'])

# Calculate cumulative deliveries per match and inning
ipl_data['delivery_count'] = ipl_data.groupby(['match_id', 'inning']).cumcount() + 1

# Calculate over number (each over = 6 deliveries)
ipl_data['calculated_over'] = ((ipl_data['delivery_count'] - 1) // 6) + 1

# Now calculate phases correctly
ipl_data['is_powerplay'] = ipl_data['calculated_over'] <= 6
ipl_data['is_death_over'] = ipl_data['calculated_over'] >= 16
ipl_data['is_middle_over'] = (ipl_data['calculated_over'] >= 7) & (ipl_data['calculated_over'] <= 15)

print(f"\n📊 Phase Distribution:")
print(f"Power Play overs (1-6): {ipl_data['is_powerplay'].sum():,} deliveries")
print(f"Middle overs (7-15): {ipl_data['is_middle_over'].sum():,} deliveries")
print(f"Death overs (16-20): {ipl_data['is_death_over'].sum():,} deliveries")

# Verify with sample
print("\n✅ Verification - First 30 deliveries with over numbers:")
sample_verify = ipl_data[['match_id', 'inning', 'ball', 'delivery_count', 'calculated_over', 'batter']].head(30)
print(sample_verify.to_string())

# Show distribution of calculated overs
print(f"\n📈 Calculated over range: {ipl_data['calculated_over'].min()} to {ipl_data['calculated_over'].max()}")
print(f"Unique overs: {sorted(ipl_data['calculated_over'].unique())}")


ADDING POWER PLAY & DEATH OVERS FLAGS (CORRECT CALCULATION)

📊 Phase Distribution:
Power Play overs (1-6): 78,830 deliveries
Middle overs (7-15): 116,354 deliveries
Death overs (16-20): 65,736 deliveries

✅ Verification - First 30 deliveries with over numbers:
     match_id  inning  ball  delivery_count  calculated_over       batter
0      335982       1     1               1                1   SC Ganguly
7      335982       1     1               2                1  BB McCullum
13     335982       1     1               3                1   SC Ganguly
19     335982       1     1               4                1  BB McCullum
26     335982       1     1               5                1   SC Ganguly
32     335982       1     1               6                1  BB McCullum
38     335982       1     1               7                2  BB McCullum
44     335982       1     1               8                2  BB McCullum
50     335982       1     1               9                2  BB McCullu

In [11]:
print("\n" + "=" * 70)
print("ADDING RUNS & WICKET METRICS")
print("=" * 70)

# Calculate metrics
ipl_data['total_runs'] = ipl_data['batsman_runs'] + ipl_data['extra_runs']
ipl_data['is_wicket'] = ipl_data['player_dismissed'].notna().astype(int)
ipl_data['is_boundary'] = (ipl_data['batsman_runs'] == 4) | (ipl_data['batsman_runs'] == 6)
ipl_data['is_dot_ball'] = (ipl_data['total_runs'] == 0) & (ipl_data['is_wicket'] == 0)

print(f"Total runs scored: {ipl_data['total_runs'].sum():,}")
print(f"Total wickets: {ipl_data['is_wicket'].sum():,}")
print(f"Boundaries: {ipl_data['is_boundary'].sum():,}")
print(f"Dot balls: {ipl_data['is_dot_ball'].sum():,}")


ADDING RUNS & WICKET METRICS
Total runs scored: 347,756
Total wickets: 12,950
Boundaries: 42,901
Dot balls: 77,945


In [12]:
print("\n" + "=" * 70)
print("SAVING CLEANED DATASETS")
print("=" * 70)

# Save to data_processed folder
ipl_data.to_csv('data_processed/ipl_clean_data.csv', index=False)
matches_clean.to_csv('data_processed/matches_clean.csv', index=False)

print("✅ Cleaned data saved to:")
print("   - data_processed/ipl_clean_data.csv")
print("   - data_processed/matches_clean.csv")
print(f"\nFinal cleaned dataset shape: {ipl_data.shape[0]:,} rows, {ipl_data.shape[1]} columns")
print(f"\n📋 Columns in cleaned dataset:")
print(ipl_data.columns.tolist())


SAVING CLEANED DATASETS
✅ Cleaned data saved to:
   - data_processed/ipl_clean_data.csv
   - data_processed/matches_clean.csv

Final cleaned dataset shape: 260,920 rows, 33 columns

📋 Columns in cleaned dataset:
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'id', 'season_year', 'season_name', 'city', 'venue', 'winner', 'toss_winner', 'toss_decision', 'is_powerplay', 'is_death_over', 'is_middle_over', 'over_num', 'delivery_count', 'calculated_over', 'is_boundary', 'is_dot_ball']


In [13]:
print("\n" + "=" * 70)
print("CREATING SQLITE DATABASE")
print("=" * 70)

# Create SQLite connection
conn = sqlite3.connect('data_processed/ipl_analytics.db')

# Load data into SQL
ipl_data.to_sql('ipl_ball_by_ball', conn, if_exists='replace', index=False)
matches_clean.to_sql('ipl_matches', conn, if_exists='replace', index=False)

print("✅ Data loaded into SQLite database!")
print(f"Table 'ipl_ball_by_ball': {pd.read_sql('SELECT COUNT(*) FROM ipl_ball_by_ball', conn).iloc[0,0]:,} rows")
print(f"Table 'ipl_matches': {pd.read_sql('SELECT COUNT(*) FROM ipl_matches', conn).iloc[0,0]:,} rows")

conn.close()
print("✅ Database saved to: data_processed/ipl_analytics.db")


CREATING SQLITE DATABASE
✅ Data loaded into SQLite database!
Table 'ipl_ball_by_ball': 260,920 rows
Table 'ipl_matches': 1,095 rows
✅ Database saved to: data_processed/ipl_analytics.db


In [14]:
print("\n" + "=" * 70)
print("KEY INSIGHTS")
print("=" * 70)

# Top batsmen
top_batsmen = ipl_data.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(10)
print("\n🏏 Top 10 Batsmen by Runs:")
print(top_batsmen)

# Top bowlers
top_bowlers = ipl_data[ipl_data['is_wicket'] == 1].groupby('bowler').size().sort_values(ascending=False).head(10)
print("\n🎯 Top 10 Bowlers by Wickets:")
print(top_bowlers)

# Team runs
team_runs = ipl_data.groupby('batting_team')['total_runs'].sum().sort_values(ascending=False).head(10)
print("\n🏆 Top 10 Teams by Total Runs:")
print(team_runs)

# Phase performance
print("\n📊 Runs by Phase:")
print(f"Power Play: {ipl_data[ipl_data['is_powerplay']]['total_runs'].sum():,} runs")
print(f"Middle Overs: {ipl_data[ipl_data['is_middle_over']]['total_runs'].sum():,} runs")
print(f"Death Overs: {ipl_data[ipl_data['is_death_over']]['total_runs'].sum():,} runs")

print("\n✅ Analysis complete! Ready for Power BI dashboard.")


KEY INSIGHTS

🏏 Top 10 Batsmen by Runs:
batter
V Kohli           8014
S Dhawan          6769
RG Sharma         6630
DA Warner         6567
SK Raina          5536
MS Dhoni          5243
AB de Villiers    5181
CH Gayle          4997
RV Uthappa        4954
KD Karthik        4843
Name: batsman_runs, dtype: int64

🎯 Top 10 Bowlers by Wickets:
bowler
YS Chahal     213
DJ Bravo      207
PP Chawla     201
SP Narine     200
R Ashwin      198
B Kumar       195
SL Malinga    188
A Mishra      183
JJ Bumrah     182
RA Jadeja     169
dtype: int64

🏆 Top 10 Teams by Total Runs:
batting_team
Mumbai Indians                 42176
Kolkata Knight Riders          39331
Chennai Super Kings            38629
Royal Challengers Bangalore    37692
Rajasthan Royals               34747
Kings XI Punjab                30064
Sunrisers Hyderabad            29071
Delhi Daredevils               24296
Delhi Capitals                 14900
Deccan Chargers                11463
Name: total_runs, dtype: int64

📊 Runs by Pha

In [15]:
print("\n" + "=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)

print(f"✅ Total matches processed: {matches_clean.shape[0]:,}")
print(f"✅ Total deliveries processed: {ipl_data.shape[0]:,}")
print(f"✅ Total runs scored: {ipl_data['total_runs'].sum():,}")
print(f"✅ Total wickets: {ipl_data['is_wicket'].sum():,}")
print(f"✅ Total boundaries: {ipl_data['is_boundary'].sum():,}")
print(f"✅ Total dot balls: {ipl_data['is_dot_ball'].sum():,}")
print(f"✅ Seasons covered: {sorted(matches_clean['season_year'].unique())}")

print("\n📁 Files created:")
print("   1. data_processed/ipl_clean_data.csv")
print("   2. data_processed/matches_clean.csv")
print("   3. data_processed/ipl_analytics.db")

print("\n🎯 Next step: Open Power BI and connect to:")
print("   - ipl_clean_data.csv (ball-by-ball data)")
print("   - matches_clean.csv (match metadata)")

print("\n🚀 Your data is ready for the Power BI dashboard!")


PROJECT SUMMARY
✅ Total matches processed: 1,095
✅ Total deliveries processed: 260,920
✅ Total runs scored: 347,756
✅ Total wickets: 12,950
✅ Total boundaries: 42,901
✅ Total dot balls: 77,945
✅ Seasons covered: [np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

📁 Files created:
   1. data_processed/ipl_clean_data.csv
   2. data_processed/matches_clean.csv
   3. data_processed/ipl_analytics.db

🎯 Next step: Open Power BI and connect to:
   - ipl_clean_data.csv (ball-by-ball data)
   - matches_clean.csv (match metadata)

🚀 Your data is ready for the Power BI dashboard!
